### Step 1: Install Required Libraries
We begin by installing the necessary Hugging Face libraries: `datasets`, `evaluate`, and `transformers` (with `sentencepiece` support).


In [ ]:
#!pip install datasets evaluate transformers[sentencepiece]

### Step 2: Define the Corpus
To understand Byte-Pair Encoding (BPE), we need a training corpus. We define a small corpus of four sentences. BPE is trained by identifying the most common character pairs in this corpus and merging them.


In [1]:
corpus = [
    "This is the Hugging Face Course.",
    "This chapter is about tokenization.",
    "This section shows several tokenizer algorithms.",
    "Hopefully, you will be able to understand how they are trained and generate tokens.",
]

### Step 3: Load a Pre-trained Tokenizer for Pre-tokenization
We load the GPT-2 tokenizer. We won't use it for the final tokenization, but we use its pre-tokenizer to split our corpus into words and handle spaces (which GPT-2 represents using the special character `Ġ`).


In [2]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")


### Step 4: Compute Word Frequencies after Pre-tokenization
We run our corpus through the GPT-2 pre-tokenizer. This splits the text into words and spaces. We then compute the frequencies of these pre-tokenized words in our corpus.


In [3]:
from collections import defaultdict

word_freqs = defaultdict(int)

for text in corpus:
    words_with_offsets = tokenizer.backend_tokenizer.pre_tokenizer.pre_tokenize_str(text)
    new_words = [word for word, offset in words_with_offsets]
    for word in new_words:
        word_freqs[word] += 1

print(word_freqs)

defaultdict(<class 'int'>, {'This': 3, 'Ġis': 2, 'Ġthe': 1, 'ĠHugging': 1, 'ĠFace': 1, 'ĠCourse': 1, '.': 4, 'Ġchapter': 1, 'Ġabout': 1, 'Ġtokenization': 1, 'Ġsection': 1, 'Ġshows': 1, 'Ġseveral': 1, 'Ġtokenizer': 1, 'Ġalgorithms': 1, 'Hopefully': 1, ',': 1, 'Ġyou': 1, 'Ġwill': 1, 'Ġbe': 1, 'Ġable': 1, 'Ġto': 1, 'Ġunderstand': 1, 'Ġhow': 1, 'Ġthey': 1, 'Ġare': 1, 'Ġtrained': 1, 'Ġand': 1, 'Ġgenerate': 1, 'Ġtokens': 1})


### Step 5: Extract the Base Alphabet
The base vocabulary of a BPE tokenizer consists of all the unique characters (letters, punctuation, and special symbols like `Ġ`) that appear in our corpus. We extract, sort, and print these base characters.


In [4]:
alphabet = []

for word in word_freqs.keys():
    for letter in word:
        if letter not in alphabet:
            alphabet.append(letter)
alphabet.sort()

print(alphabet)

[',', '.', 'C', 'F', 'H', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z', 'Ġ']


### Step 6: Initialize the Vocabulary
We initialize our tokenizer's vocabulary. It starts with the special token `<|endoftext|>` (used by GPT-2 to mark the end of text) followed by the base alphabet we extracted in the previous step.


In [5]:
vocab = ["<|endoftext|>"] + alphabet.copy()

### Step 7: Represent Words as Lists of Characters (Splits)
We split every word in our word frequency dictionary into its individual characters. As BPE runs, we will progressively merge these characters into subwords based on frequency.


In [6]:
splits = {word: [c for c in word] for word in word_freqs.keys()}

### Step 8: Function to Compute Symbol Pair Frequencies
We define `compute_pair_freqs`, which counts the frequency of all adjacent symbol pairs in our current word splits across the entire corpus.


In [7]:
def compute_pair_freqs(splits):
    pair_freqs = defaultdict(int)
    for word, freq in word_freqs.items():
        split = splits[word]
        if len(split) == 1:
            continue
        for i in range(len(split) - 1):
            pair = (split[i], split[i + 1])
            pair_freqs[pair] += freq
    return pair_freqs

### Step 9: Get Initial Pair Frequencies
We call `compute_pair_freqs` on our initial splits and print the first few pair frequencies to see which character sequences are adjacent.


In [8]:
pair_freqs = compute_pair_freqs(splits)

for i, key in enumerate(pair_freqs.keys()):
    print(f"{key}: {pair_freqs[key]}")
    if i >= 5:
        break

('T', 'h'): 3
('h', 'i'): 3
('i', 's'): 5
('Ġ', 'i'): 2
('Ġ', 't'): 7
('t', 'h'): 3


### Step 10: Find the Most Frequent Pair
We search through all symbol pairs to find the one with the highest frequency. In our corpus, the most frequent pair is `('Ġ', 't')` with a frequency of 7.


In [9]:
best_pair = ""
max_freq = None

for pair, freq in pair_freqs.items():
    if max_freq is None or max_freq < freq:
        best_pair = pair
        max_freq = freq

print(best_pair, max_freq)

('Ġ', 't') 7


### Step 11: Register the First Merge Rule
We create our first merge rule, mapping `('Ġ', 't')` to `"Ġt"`, and add the new merged token `"Ġt"` to our vocabulary.


In [10]:
merges = {("Ġ", "t"): "Ġt"}
vocab.append("Ġt")

### Step 12: Verify Vocabulary Update
We display the current vocabulary to confirm that our first merged token `"Ġt"` has been added.


In [11]:
vocab

['<|endoftext|>',
 ',',
 '.',
 'C',
 'F',
 'H',
 'T',
 'a',
 'b',
 'c',
 'd',
 'e',
 'f',
 'g',
 'h',
 'i',
 'k',
 'l',
 'm',
 'n',
 'o',
 'p',
 'r',
 's',
 't',
 'u',
 'v',
 'w',
 'y',
 'z',
 'Ġ',
 'Ġt']

### Step 13: Define the `merge_pair` Function
This is the core function of BPE training that performs the merge operation. Let's break down how it works:

1. **Loop through words:** We loop through each unique word in the corpus.
2. **Skip single characters:** If a word split has a length of 1, it cannot contain any pairs, so we skip it.
3. **Scan and replace (while loop):** We scan the list using index pointer `i` from `0` to `len(split) - 2`:
   - If we find the pair `[a, b]` at indices `i` and `i + 1` (i.e., `split[i] == a` and `split[i + 1] == b`), we merge them.
   - **Merging operation:** We slice the list: `split[:i]` (everything before index `i`), plus `[a + b]` (the new merged token), plus `split[i + 2:]` (everything after the pair).
   - **Pointer increment:** We do **not** increment `i` when a merge occurs. This is because the list shrunk, and the elements shifted left. The element that was at `i + 2` is now at `i + 1`, and the newly merged element is at `i`. So we need to re-evaluate the elements from index `i` (specifically, check if the newly merged element and its new neighbor form a valid pair next).
   - If the pair does not match, we simply increment `i` by 1 to inspect the next position.
4. **Update split:** We update the word split in our dictionary `splits` and return it.


In [13]:
def merge_pair(a, b, splits):
    for word in word_freqs:
        split = splits[word]
        if len(split) == 1:
            continue

        i = 0
        while i < len(split) - 1:
            if split[i] == a and split[i + 1] == b:
                split = split[:i] + [a + b] + split[i + 2 :]
            else:
                i += 1
        splits[word] = split
    return splits

### Step 14: Apply and Verify the First Merge
We apply `merge_pair` to merge the pair `("Ġ", "t")` across all our word splits. We print the split for `"Ġtrained"` to verify that it is now `['Ġt', 'r', 'a', 'i', 'n', 'e', 'd']` instead of starting with `['Ġ', 't', ...]`.


In [14]:
splits = merge_pair("Ġ", "t", splits)
print(splits["Ġtrained"])

['Ġt', 'r', 'a', 'i', 'n', 'e', 'd']


### Step 15: Run BPE Training Loop
We define a target vocabulary size of 50. In a loop, we repeatedly compute the frequencies of all adjacent pairs, find the most frequent pair, apply `merge_pair` to merge it in all splits, record the merge rule, and add the new subword token to the vocabulary.


In [15]:
vocab_size = 50

while len(vocab) < vocab_size:
    pair_freqs = compute_pair_freqs(splits)
    best_pair = ""
    max_freq = None
    for pair, freq in pair_freqs.items():
        if max_freq is None or max_freq < freq:
            best_pair = pair
            max_freq = freq
    splits = merge_pair(*best_pair, splits)
    merges[best_pair] = best_pair[0] + best_pair[1]
    vocab.append(best_pair[0] + best_pair[1])

### Step 16: View Learned Merge Rules
We print the `merges` dictionary to see all the merge rules learned by our BPE tokenizer in order. These rules dictate how we will tokenize unseen text.


In [16]:
print(merges)

{('Ġ', 't'): 'Ġt', ('i', 's'): 'is', ('e', 'r'): 'er', ('Ġ', 'a'): 'Ġa', ('Ġt', 'o'): 'Ġto', ('e', 'n'): 'en', ('T', 'h'): 'Th', ('Th', 'is'): 'This', ('o', 'u'): 'ou', ('s', 'e'): 'se', ('Ġto', 'k'): 'Ġtok', ('Ġtok', 'en'): 'Ġtoken', ('n', 'd'): 'nd', ('Ġ', 'is'): 'Ġis', ('Ġt', 'h'): 'Ġth', ('Ġth', 'e'): 'Ġthe', ('i', 'n'): 'in', ('Ġa', 'b'): 'Ġab', ('Ġtoken', 'i'): 'Ġtokeni'}


### Step 17: View Final BPE Vocabulary
We print the final vocabulary of 50 tokens to see the base characters along with all the newly created subwords.


In [17]:
print(vocab)

['<|endoftext|>', ',', '.', 'C', 'F', 'H', 'T', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'k', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'v', 'w', 'y', 'z', 'Ġ', 'Ġt', 'is', 'er', 'Ġa', 'Ġto', 'en', 'Th', 'This', 'ou', 'se', 'Ġtok', 'Ġtoken', 'nd', 'Ġis', 'Ġth', 'Ġthe', 'in', 'Ġab', 'Ġtokeni']


### Step 18: Define the `tokenize` Function for New Text
To tokenize new text using our trained BPE model:
1. **Pre-tokenize:** Use the GPT-2 pre-tokenizer to split text into words and spaces.
2. **Character splits:** Split each pre-tokenized word into individual characters.
3. **Apply merges in order:** Iterate through our learned merge rules in the exact order they were trained, merging matching pairs within the splits.
4. **Flatten:** Combine the splits of all words into a single flat list of tokens.


In [18]:
def tokenize(text):
    pre_tokenize_result = tokenizer._tokenizer.pre_tokenizer.pre_tokenize_str(text)
    pre_tokenized_text = [word for word, offset in pre_tokenize_result]
    splits = [[l for l in word] for word in pre_tokenized_text]
    for pair, merge in merges.items():
        for idx, split in enumerate(splits):
            i = 0
            while i < len(split) - 1:
                if split[i] == pair[0] and split[i + 1] == pair[1]:
                    split = split[:i] + [merge] + split[i + 2 :]
                else:
                    i += 1
            splits[idx] = split

    return sum(splits, [])

### Step 19: Test Tokenization on a New Sentence
We test our tokenizer on an unseen sentence. Notice how common subwords like `"er"`, `"en"`, and `"Ġ"` are grouped, while unseen patterns are split into individual letters.


In [19]:
tokenize("I will be mastering generative AI")

['I',
 'Ġ',
 'w',
 'i',
 'l',
 'l',
 'Ġ',
 'b',
 'e',
 'Ġ',
 'm',
 'a',
 's',
 't',
 'er',
 'in',
 'g',
 'Ġ',
 'g',
 'en',
 'er',
 'a',
 't',
 'i',
 'v',
 'e',
 'Ġ',
 'A',
 'I']

### Step 20: Test Tokenization on a Sentence containing Learned Vocabulary
We tokenize another sentence containing words that match our learned vocabulary. Because these subwords are in our vocabulary and merges list, they are successfully tokenized as complete units (e.g., `"This"`, `"Ġis"`, `"Ġtoken"`).


In [20]:
tokenize("This is not a token.")

['This', 'Ġis', 'Ġ', 'n', 'o', 't', 'Ġa', 'Ġtoken', '.']